# L04 · Roles causales y datos de Jobs

**S4 · Lectura: Cap. 4 · 40–55 min**

## La pregunta

¿Qué columna es tratamiento, resultado y covariable?

## Qué harás

Declarar T, Y y X antes de estimar.

- Lee la idea breve.
- Ejecuta la demostración y mira el gráfico.
- Completa las celdas marcadas.
- Escribe qué significa el resultado y qué no permite afirmar.

**Producto:** Un esquema validado y una primera tabla de balance en Jobs.

> **Idea clave:** El rol de una variable depende de la pregunta y del momento en que se mide.

La solución está en `la ruta de rescate docente (no incluida)`. Intenta primero el notebook del estudiante.

In [ ]:
# Localiza lab_common.py (carpeta actual o una superior)
import sys, pathlib
for _p in [pathlib.Path.cwd(), *list(pathlib.Path.cwd().parents)[:3]]:
    if (_p / "lab_common.py").exists():
        sys.path.insert(0, str(_p)); break

## 0. Preparación

In [ ]:
from lab_common import *
from lab_assumptions import smd
import numpy as np, pandas as pd
from dataclasses import dataclass, field
rng = setup()
sc = load_scenario("real/jobs")
sc.card()

In [ ]:
# Vista rápida de los datos del escenario
import matplotlib.pyplot as plt
import numpy as np
_fig, _ax = plt.subplots(1, 2, figsize=(9, 3.2))
try:
    _scenario = globals().get("sc") or globals().get("lc")
    if _scenario is None:
        raise NameError("No hay un escenario cargado para el gráfico editorial")
    _df = _scenario.train
    if _scenario.treatment is not None and _scenario.outcome is not None:
        _t = _df[_scenario.treatment].to_numpy(float)
        _y = _df[_scenario.outcome].to_numpy(float)
        if np.unique(_t).size == 2:
            _counts = [int((_t == value).sum()) for value in (0, 1)]
            _ax[0].bar(["control", "tratado"], _counts)
            _ax[0].set_title("Asignación")
            _ax[1].boxplot([_y[_t == 0], _y[_t == 1]], labels=["control", "tratado"])
            _ax[1].set_title("Resultado observado")
        else:
            _ax[0].hist(_t, bins=20)
            _ax[0].set_title(_scenario.treatment)
            _ax[1].scatter(_t, _y, s=8, alpha=.35)
            _ax[1].set_title(f"{_scenario.treatment} frente a {_scenario.outcome}")
    elif _scenario.adj is not None:
        _ax[0].imshow(_scenario.adj, cmap="Greys", interpolation="nearest")
        _ax[0].set_title("Grafo del escenario")
        _ax[1].axis("off")
    else:
        _ax[0].axis("off"); _ax[1].axis("off")
    _fig.suptitle(f"Escenario: {_scenario.name}")
    _fig.tight_layout(); plt.show()
finally:
    plt.close(_fig)

## 1. Por qué los roles van en un objeto
En un modelo predictivo cualquier columna puede ser predictor. En uno causal **el rol de cada columna decide qué se estima**: el tratamiento $D$, el resultado $Y$ y las covariables $X$ que se *miden antes* del tratamiento. Si los roles viven en variables sueltas del notebook, nadie puede auditarlos ni servirlos después. La decisión del capítulo es declararlos en un `CausalSchema` que se **valida** y viaja junto al modelo.

In [ ]:
@dataclass
class CausalSchema:
    treatment: str
    outcome: str
    covariates: list
    instruments: list = field(default_factory=list)

    def validate(self, df):
        roles = [self.treatment, self.outcome, *self.covariates, *self.instruments]
        faltan = [c for c in roles if c not in df.columns]
        if faltan: raise ValueError(f"Columnas inexistentes: {faltan}")
        dup = {c for c in roles if roles.count(c) > 1}
        if dup: raise ValueError(f"Una columna no puede tener dos roles: {sorted(dup)}")
        if df[self.treatment].dropna().nunique() != 2: raise ValueError("El tratamiento debe ser binario")
        if df[[self.treatment, self.outcome]].isna().any().any(): raise ValueError("Tratamiento/resultado con faltantes")
        assert_no_truth(df[self.covariates], "covariables")
        return True

    def X(self, df): return df[self.covariates].to_numpy(float)
    def D(self, df): return df[self.treatment].to_numpy(float)
    def Y(self, df): return df[self.outcome].to_numpy(float)

schema = CausalSchema(treatment=sc.treatment, outcome=sc.outcome, covariates=list(sc.covariates))
print(schema)
print("validate →", schema.validate(sc.train))

## 2. Provoca los errores típicos

In [ ]:
casos = {
    "resultado entre las covariables": CausalSchema(sc.treatment, sc.outcome, [*sc.covariates, sc.outcome]),
    "tratamiento entre las covariables": CausalSchema(sc.treatment, sc.outcome, [*sc.covariates, sc.treatment]),
    "columna inexistente": CausalSchema(sc.treatment, sc.outcome, [*sc.covariates, "ingreso_futuro"]),
}
for nombre, s in casos.items():
    try:
        s.validate(sc.train); print(f"✘ NO detectado: {nombre}")
    except ValueError as e:
        print(f"✔ detectado: {nombre:36s} → {e}")

**Mira el resultado.** ¿Qué error de roles **no** puede detectar `validate` por sí solo?

**Respuesta:**

Una covariable **posterior al tratamiento** con un nombre inocente (p. ej. `ingreso_6m`): estructuralmente es válida. Hace falta *conocimiento de cuándo se midió* cada columna; lo vemos en L08 (Cap. 9).



## 3. `summary()` por rol (ejercicio del capítulo)
Una auditoría rápida antes de estimar: por cada rol, cuántas columnas hay, el tipo de dato y los faltantes.

In [ ]:
# ✏️ Completa tú — summary
# Pista: df[c].dtype y df[c].isna().sum()
def summary(self, df):
    """DataFrame con columnas: rol, columna, dtype, faltantes (una fila por columna con rol)."""
    raise NotImplementedError("recorre los cuatro roles y reporta dtype y número de faltantes")
CausalSchema.summary = summary

In [ ]:
resumen = schema.summary(sc.train)
show(resumen)
assert list(resumen.columns) == ["rol", "columna", "dtype", "faltantes"] and len(resumen) == 2 + len(sc.covariates)
print("faltantes totales:", int(resumen.faltantes.sum()))

## 4. Mira los datos antes de estimar (A1)
Tasa de tratamiento, diferencia de medias descriptiva y balance (diferencia media estandarizada, SMD) de las covariables. El SMD se estudia a fondo en L06; aquí solo úsalo como radiografía.

In [ ]:
X, y, d = schema.X(sc.train), schema.Y(sc.train), schema.D(sc.train)
est, se = diff_in_means(y, d)
print(f"tasa de tratamiento = {d.mean():.4f}   diferencia en medias = {fmt_ci(est, se, 1)}")
bal = pd.DataFrame({"covariable": sc.covariates, "SMD": [smd(X[:, j], d) for j in range(X.shape[1])]})
bal["|SMD|"] = bal.SMD.abs()
show(bal.sort_values("|SMD|", ascending=False).round(3))

In [ ]:
import matplotlib.pyplot as plt
b = bal.sort_values("|SMD|")
plt.barh(b.covariable, b["|SMD|"]); plt.axvline(.1, ls="--", c="gray"); plt.axvline(.25, ls=":", c="red")
plt.title("|SMD| por covariable (guías en 0,1 y 0,25)"); plt.show()

**Mira el resultado.** La diferencia en medias es una *descripción*, no un efecto. ¿Qué tendría que cumplirse para leerla como efecto?

**Respuesta:**

Que, dadas las covariables medidas, la asignación fuera como aleatoria (no confusión condicional). Eso es un supuesto **no observable** que se declara (L05–L06).



## 5. Supuesto y bitácora

In [ ]:
fila = bitacora(
    pregunta="¿Están bien definidos los roles causales del caso A1 y qué se ve a simple vista en los datos?",
    metodo="CausalSchema validado + diferencia en medias descriptiva + SMD",
    estimando="Diferencia en medias descriptiva (no efecto causal)",
    numero_ic=f"{fmt_ci(est, se, 1)}; SMD máximo = {bal['|SMD|'].max():.3f}",
    supuesto=supuesto("observable", "medido", f"SMD máximo = {bal['|SMD|'].max():.3f} entre tratados y controles"),
    decision="continuar", lab="L04")

---
## Qué queda para el libro
Ejercicio del libro cubierto: añadir `summary()` y aplicarlo al dataset de A1. El callout del monorepo remite a `lib/schema.py`; aquí el schema se define dentro del lab para que sea autocontenido.

*Material formativo (sin nota). Los datos son sintéticos: ver la ficha del escenario.*